```Student name: Pham Ngoc Hao```

```Student ID: 23110146```

DISCUSSION QUESTIONS

- Why does Transformer not need recurrence?

Transformers eliminate the need for recurrence by replacing sequential processing with a Self-Attention mechanism. This shift addresses two major limitations of recurrent models (RNNs/LSTMs):

1. Parallelization: Recurrent models must process tokens one by one (time step $t$ depends on $t-1$), which prevents efficient use of GPU parallel computing. Transformers process the entire sequence simultaneously, significantly speeding up training.

2. Long-Range Dependencies: In RNNs, information must pass through many hidden states, often leading to the vanishing gradient problem or a "bottleneck" where context from the beginning of a sentence is lost. Self-attention allows every word to interact directly with every other word, regardless of their distance, preserving global context perfectly.

- Why is positional encoding necessary?

In a Transformer, the Self-Attention mechanism processes all tokens in a sequence simultaneously (in parallel). Unlike RNNs, which process words one by one, the Transformer has no inherent sense of the "order" or "position" of words.

Positional Encoding is necessary because:

1. Breaking Permutation Invariance: Without it, the model would treat the sentence "The cat ate the fish" exactly the same as "The fish ate the cat". The attention scores would be identical because the "bag of words" is the same.

2. Adding Sequential Context: It injects a unique signal into each word's embedding, allowing the model to distinguish where a word sits in the sequence.

3. Handling Distance: It helps the model learn that words closer to each other might have different relationships than words far apart.

- Why does multi-head attention help?

Multi-head attention is beneficial because it allows the model to jointly attend to information from different representation subspaces at different positions.

1. Capturing Diverse Relationships: Instead of having one attention weighted average, multiple heads can capture different types of relationships (e.g., one head for syntax/grammar, another for semantic meaning, and another for long-range dependencies).

2. Stability: It functions similarly to an "ensemble" of observers, making the model’s learning process more stable and robust against noise in the data.

3. Feature Richness: By projecting the original embeddings into smaller subspaces, each head can focus on a specific feature set without being overwhelmed by the entire input vector.

- What happens if we remove attention?

If the attention mechanism is removed, the Transformer effectively loses its ability to process sequences as a cohesive unit. The specific consequences include:

1. Loss of Contextual Representation: Every word would be treated with equal weight, making the model unable to distinguish which words are most relevant to the current token.

2. Blindness to Long-Range Dependencies: The model would only "see" local information, failing to connect related concepts that are far apart in a sentence.

3. Architectural Failure: Since the Transformer processes the entire sentence in parallel, Attention is the only mechanism that allows for information exchange between positions. Without it, the model collapses into a series of independent operations on individual words, losing all structural and grammatical understanding.

- How is attention different from simple averaging?

The fundamental difference lies in how information is aggregated across the sequence:

1. Simple Averaging: It treats all tokens as equally important by assigning a fixed weight of $1/n$ to each. This results in a "loss of focus," as noise or filler words contribute as much as critical keywords to the final representation.

2. Attention Mechanism: It calculates dynamic weights based on the relevance (similarity) between words. By focusing on the most informative parts of the sentence and suppressing irrelevant ones, the model creates a context-aware representation that is much more powerful for understanding complex language structures.

# Experiment Task

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk
import re
import math
import copy
import torch.optim as optim
import random

from tqdm.auto import tqdm
from datasets import load_dataset
from dataclasses import dataclass
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

In [ ]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [ ]:
@dataclass
class Config:
  DEBUG: bool = True
  LEARNING_RATE: float = 3e-4
  SEQ_LENGTH: int = 64
  MAX_LEN: int = 64
  BATCH_SIZE: int = 64
  EMBED_DIM: int = 256
  NUM_HEADS: int = 8
  NUM_LAYERS: int = 4
  HIDDEN_DIM: int = 512
  DROPOUT: float = 0.1

  def __post_init__(self):
    if self.DEBUG:
      self.EPOCHS=1
      self.NUM_SAMPLES=100
    else:
      self.EPOCHS=50
      self.NUM_SAMPLES=10000
    self.DEVICE= torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Step 1 - Clean the data
Raw text data (e.g., from WikiText) often contains:

- Empty lines
- Extra spaces
- Inconsistent formatting → Therefore, we first clean the data.

What we do:

- Remove empty lines
- Strip unnecessary spaces
- Convert all text to lowercase

Question: Why this matters?

Data cleaning is a crucial first step for several reasons:

1. Vocabulary Efficiency: By converting to lowercase and removing noise, we significantly reduce the vocabulary size. This prevents the model from treating "Apple" and "apple" as different entities, allowing for more efficient embedding learning.

2. Focusing Attention: Removing extra spaces and empty lines ensures that the Transformer’s attention mechanism focuses on meaningful tokens rather than structural noise.

3. Consistency: Standardizing the text format helps the model identify linguistic patterns more easily, leading to faster convergence and better performance during training.

In [ ]:
def clean_data(sentence: str) -> list[str]:
  sentence = re.sub(r"[^\w\s]", " ", str(sentence))
  sentence = re.sub(r"\s+"," ", sentence)
  sentence = sentence.lower()
  return sentence.split()

## Step 2 - Build a corpus

After cleaning, we combine all sentences into one long sequence of text.

Process:

- Join multiple sentences into a single string
- Split into individual words (tokens)
Important note:

We only use a subset of the dataset (e.g., first 10000 lines)

In [ ]:
def build_corpus(data: list[str]) -> list[str]:
  corpus = []
  for text in data:
    if text.strip():
      tokens = clean_data(text)
      corpus.extend(tokens)
  return corpus

In [ ]:
def test_corpus():
  corpus = build_corpus(data=dataset["train"]["text"][:5])
  print(corpus)
  print(len(corpus))

In [ ]:
test_corpus()

['valkyria', 'chronicles', 'iii', 'senjō', 'no', 'valkyria', '3', 'unrecorded', 'chronicles', 'japanese', '戦場のヴァルキュリア3', 'lit', 'valkyria', 'of', 'the', 'battlefield', '3', 'commonly', 'referred', 'to', 'as', 'valkyria', 'chronicles', 'iii', 'outside', 'japan', 'is', 'a', 'tactical', 'role', 'playing', 'video', 'game', 'developed', 'by', 'sega', 'and', 'media', 'vision', 'for', 'the', 'playstation', 'portable', 'released', 'in', 'january', '2011', 'in', 'japan', 'it', 'is', 'the', 'third', 'game', 'in', 'the', 'valkyria', 'series', 'employing', 'the', 'same', 'fusion', 'of', 'tactical', 'and', 'real', 'time', 'gameplay', 'as', 'its', 'predecessors', 'the', 'story', 'runs', 'parallel', 'to', 'the', 'first', 'game', 'and', 'follows', 'the', 'nameless', 'a', 'penal', 'military', 'unit', 'serving', 'the', 'nation', 'of', 'gallia', 'during', 'the', 'second', 'europan', 'war', 'who', 'perform', 'secret', 'black', 'operations', 'and', 'are', 'pitted', 'against', 'the', 'imperial', 'unit', 'ca

## Step 3 - Create training data

We convert the text into a next-word prediction task.

Idea:

- Given a sequence of words → predict the next word
Example: Input: [w1, w2, w3] Target: w4

Intuition:

- The model learns context
- It understands how words follow each other

In [ ]:
def create_training_data(corpus: list[str], word2idx: dict[str, int], seq_length: int = 5) -> tuple[torch.Tensor, torch.Tensor]:
  inputs = []
  targets = []

  for i in range(len(corpus) - seq_length):
    input_seq = corpus[i : i + seq_length]

    target = corpus[i + seq_length]

    inputs.append([
        word2idx.get(w, word2idx["<UNK>"])
        for w in input_seq
    ])

    targets.append(
        word2idx.get(target, word2idx["<UNK>"])
    )
  X = torch.LongTensor(inputs)
  y = torch.LongTensor(targets)
  return X, y


## Step 4 - Build Vocabulary

We create a mapping between words and numbers.

Two mappings:

- word → index (for input) ("i love nlp" → [3, 10, 25])
- index → word (for decoding output)

In [ ]:
def build_vocab(corpus, max_vocab_size=10000):

    counter = Counter(corpus)

    most_common_words = counter.most_common(max_vocab_size)

    vocab = ["<PAD>", "<UNK>"] + [
        word for word, _ in most_common_words
    ]

    word2idx = {
        word: i
        for i, word in enumerate(vocab)
    }

    idx2word = {
        i: word
        for i, word in enumerate(vocab)
    }

    return word2idx, idx2word

In [ ]:
def test_vocab():
  corpus = ["i", "love", "nlp", "i"]
  word2idx, idx2word = build_vocab(corpus)
  print("Corpus:")
  print(corpus)

  print("\nword2idx:")
  print(word2idx)

  print("\nidx2word:")
  print(idx2word)

  print("\nVocabulary size:")
  print(len(word2idx))

In [ ]:
test_vocab()

Corpus:
['i', 'love', 'nlp', 'i']

word2idx:
{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'nlp': 4}

idx2word:
{0: '<PAD>', 1: '<UNK>', 2: 'i', 3: 'love', 4: 'nlp'}

Vocabulary size:
5


## Step 5 - Model

```python
Input (token ids)
→ Embedding (after embedding→ [[0.2, ...], [0.5, ...], [0.1, ...]])
→ Positional Encoding (Final input = embedding + positional_encoding)
→ Transformer Block:
    → Multi-head Self-Attention
        Q, K, V = Linear(X)
        scores = (Q @ K^T) / sqrt(d_k)
        weights = softmax(scores)
        attention_output = weights @ V
        concat heads → Linear
    → Add & Norm
        x = LayerNorm(x + attention_output)
    → Feed Forward
        ff_output = Linear → ReLU → Linear
    → Add & Norm
        x = LayerNorm(x + ff_output)

→ Pooling / Last Token
→ Linear Layer
→ Softmax
→ Output (prediction)

In [ ]:
class TransformerModel(nn.Module):
  def __init__(
      self,
      vocab_size: int,
      embed_dim: int = 128,
      num_heads: int = 4,
      hidden_dim: int = 256,
      num_layers: int = 4,
      max_len: int = 100,
      dropout: float = 0.3
  ):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings=vocab_size,
        embedding_dim=embed_dim
    )

    self.pos_embedding = nn.Embedding(
        num_embeddings=max_len,
        embedding_dim=embed_dim
    )

    encoder_layer = nn.TransformerEncoderLayer(
        d_model=embed_dim,
        nhead=num_heads,
        dim_feedforward=hidden_dim,
        dropout=dropout,
        batch_first=True
    )

    self.transformer = nn.TransformerEncoder(
        encoder_layer=encoder_layer,
        num_layers=num_layers
    )

    self.lm_head = nn.Linear(
        embed_dim,
        vocab_size
    )

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    batch_size, seq_len = x.shape

    positions = torch.arange(
        0,
        seq_len,
        device=x.device
    )

    positions = positions.unsqueeze(0).expand(batch_size, seq_len)

    x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
    x = x + self.pos_embedding(positions)
    x = self.dropout(x)
    mask = torch.triu(
        torch.ones(seq_len, seq_len, device=x.device),
        diagonal=1
    ).bool()
    x = self.transformer(
        x,
        mask=mask
    )

    x = x[:, -1, :]
    logits = self.lm_head(x)
    return logits

## Step 6 - Training the model

In [ ]:
class Trainer:
    def __init__(self, model, optimizer, scheduler, criterion, device, patience=5):

        self.model = model.to(device)

        self.optimizer = optimizer

        self.scheduler = scheduler

        self.criterion = criterion

        self.device = device

        self.patience = patience

        self.counter = 0

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_acc': []
        }

        self.best_val_loss = float("inf")

        self.best_model_state = None

    def train_step(self, batch_x, batch_y):

        self.model.train()

        batch_x = batch_x.to(self.device)

        batch_y = batch_y.to(self.device)

        self.optimizer.zero_grad()

        logits = self.model(batch_x)

        loss = self.criterion(logits, batch_y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.model.parameters(),
            max_norm=1.0
        )

        self.optimizer.step()

        return loss.item()

    def evaluate(self, loader):

        self.model.eval()

        total_loss = 0

        correct = 0

        total = 0

        with torch.no_grad():

            for batch_x, batch_y in loader:

                batch_x = batch_x.to(self.device)

                batch_y = batch_y.to(self.device)

                logits = self.model(batch_x)

                loss = self.criterion(logits, batch_y)

                total_loss += loss.item()

                preds = torch.argmax(logits, dim=1)

                correct += (preds == batch_y).sum().item()

                total += batch_y.size(0)

        avg_loss = total_loss / len(loader)

        accuracy = correct / total

        return avg_loss, accuracy

    def fit(self, train_loader, val_loader, epochs):

        print(f"Training on: {self.device}")

        for epoch in range(epochs):

            total_train_loss = 0

            pbar = tqdm(
                train_loader,
                desc=f"Epoch {epoch+1}/{epochs}"
            )

            for batch_x, batch_y in pbar:

                loss = self.train_step(batch_x, batch_y)

                total_train_loss += loss

                pbar.set_postfix({
                    "loss": f"{loss:.4f}"
                })

            avg_train_loss = total_train_loss / len(train_loader)

            avg_val_loss, val_accuracy = self.evaluate(val_loader)

            self.scheduler.step(avg_val_loss)

            self.history['train_loss'].append(avg_train_loss)

            self.history['val_loss'].append(avg_val_loss)

            self.history['val_acc'].append(val_accuracy)

            if avg_val_loss < self.best_val_loss:

                self.best_val_loss = avg_val_loss

                self.best_model_state = copy.deepcopy(
                    self.model.state_dict()
                )

                self.counter = 0

                print(
                    f"Saved Best Model "
                    f"(best val loss: {avg_val_loss:.4f})"
                )

            else:

                self.counter += 1

                print(
                    f"No improvement "
                    f"({self.counter}/{self.patience})"
                )

                if self.counter >= self.patience:

                    print("Early stopping triggered!")

                    break

            print(
                f"Epoch {epoch+1} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {val_accuracy:.4f} | "
                f"LR: {self.optimizer.param_groups[0]['lr']:.6f}"
            )

    def load_best_model(self):

        if self.best_model_state:

            self.model.load_state_dict(
                self.best_model_state
            )
            print("Loaded best model")

In [ ]:
def predict_next_word(model, text, word2idx, idx2word, config):
    model.eval()
    tokens = clean_data(text)

    if len(tokens) < config.SEQ_LENGTH:
        print("Warning: The input sequence is too short!")
        return None

    input_tokens = tokens[-config.SEQ_LENGTH:]

    input_ids = [
      word2idx.get(w, word2idx["<UNK>"])
      for w in input_tokens
    ]

    with torch.no_grad():
        x_input = torch.LongTensor([input_ids]).to(config.DEVICE)
        logits = model(x_input)
        pred_idx = torch.argmax(logits, dim=1).item()

    return idx2word[pred_idx]

In [ ]:
def evaluate_on_test(trainer, test_dataset, word2idx, config):
    print("Evaluate on test data")

    test_corpus = build_corpus(test_dataset)

    X_test, y_test = create_training_data(test_corpus, word2idx, config.SEQ_LENGTH)

    test_ds = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE)

    test_loss, test_accuracy = trainer.evaluate(test_loader)

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

    return test_accuracy

In [ ]:
def main():
  random.seed(42)
  torch.manual_seed(42)

  if torch.cuda.is_available():
      torch.cuda.manual_seed_all(42)

  cfg = Config(DEBUG=False)
  num_samples = cfg.NUM_SAMPLES

  train_data = dataset["train"]["text"][:num_samples]
  valid_data = dataset["validation"]["text"][:num_samples]
  test_data = dataset["test"]["text"][:num_samples]


  train_corpus = build_corpus(train_data)
  word2idx, idx2word = build_vocab(train_corpus)
  vocab_size = len(word2idx)
  print(f"Vocab size: {vocab_size}")
  val_corpus = build_corpus(valid_data)

  x_train, y_train = create_training_data(train_corpus, word2idx, cfg.SEQ_LENGTH)
  x_val, y_val = create_training_data(val_corpus, word2idx, cfg.SEQ_LENGTH)

  train_ds = TensorDataset(x_train, y_train)
  val_ds = TensorDataset(x_val, y_val)

  train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True)
  val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE)

  model = TransformerModel(
      vocab_size=vocab_size,
      embed_dim=cfg.EMBED_DIM,
      num_heads=cfg.NUM_HEADS,
      hidden_dim=cfg.HIDDEN_DIM,
      num_layers=cfg.NUM_LAYERS,
      max_len=cfg.MAX_LEN,
      dropout=cfg.DROPOUT
  )

  optimizer = optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, betas=(0.9, 0.98), weight_decay=0.01, eps=1e-8)
  scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
  criterion=nn.CrossEntropyLoss()

  trainer = Trainer(model, optimizer, scheduler, criterion, cfg.DEVICE)

  trainer.fit(train_loader, val_loader, epochs=cfg.EPOCHS)

  trainer.load_best_model()

  test_accuracy = evaluate_on_test(trainer, test_data, word2idx, cfg)
  print(f"Final Test Accuracy: {test_accuracy*100:.2f}%")

  test_corpus = build_corpus(test_data)
  print("Check 10 random sentences:")
  for i in range(10):
    idx = random.randint(0, len(test_corpus) - cfg.SEQ_LENGTH - 1)
    input_tokens = test_corpus[idx : idx + cfg.SEQ_LENGTH]
    input_sentence = " ".join(input_tokens)
    ground_truth = test_corpus[idx + cfg.SEQ_LENGTH]
    prediction = predict_next_word(model, input_sentence, word2idx, idx2word, cfg)
    print(f"# Sentence {i + 1}:")
    print(f"Original sentence: {input_sentence}")
    print(f"Ground Truth: {ground_truth}")
    print(f"Predict: {prediction}")
    print()

In [ ]:
if __name__ == "__main__":
  main()

Vocab size: 10002
Training on: cuda


Epoch 1/50:   0%|          | 0/7365 [00:00<?, ?it/s]

Saved Best Model (best val loss: 6.0413)
Epoch 1 | Train Loss: 6.5655 | Val Loss: 6.0413 | Val Acc: 0.1528 | LR: 0.000300


Epoch 2/50:   0%|          | 0/7365 [00:00<?, ?it/s]

Saved Best Model (best val loss: 5.9245)
Epoch 2 | Train Loss: 6.2088 | Val Loss: 5.9245 | Val Acc: 0.1593 | LR: 0.000300


...

No improvement (3/5)
Epoch 8 | Train Loss: 6.0396 | Val Loss: 5.9726 | Val Acc: 0.1863 | LR: 0.000150


Epoch 9/50:   0%|          | 0/7365 [00:00<?, ?it/s]

No improvement (4/5)
Epoch 9 | Train Loss: 5.9285 | Val Loss: 5.9654 | Val Acc: 0.1882 | LR: 0.000150


Epoch 10/50:   0%|          | 0/7365 [00:00<?, ?it/s]

No improvement (5/5)
Early stopping triggered!
Loaded best model
Evaluate on test data
Test Loss: 5.7263
Test Accuracy: 0.1800
Final Test Accuracy: 18.00%
Check 10 random sentences:
# Sentence 1:
Original sentence: album with producer nick raskulinecz at dave grohl s personal studio studio 606 in los angeles time in the studio began with a week of pre production during which guitarist josh rand says producer raskulinecz pushed the band to the brink and back to help fine tune the songs they had previously written though rand and taylor wrote most of the music and lyrics
Ground Truth: for
Predict: were

# Sentence 2:
Original sentence: protection their weapons were more effective than that of gloire and with the largest set of steam engines yet fitted to a ship they could steam at 14 3 knots 26 5 km h yet the gloire and her sisters had full iron armour protection along the waterline and the battery itself warrior and black prince but also the smaller defence and resistance were
Ground Tr

## Step 7: Report and explain the pipeline

# LAB 05: MINI TRANSFORMER FOR NEXT-WORD PREDICTION
**Student Name:** Pham Ngoc Hao
**Student ID:** 23110146  


## 1. Problem Description
* **Problem:** The objective is to build a language model capable of predicting the most probable next token in a sequence.
* **Input:** A fixed-length sequence of tokens (sliding window) represented as integer IDs: $X = [w_1, w_2, \dots, w_n]$, with $n=64$ in this configuration.
* **Output:** A probability distribution over the vocabulary for the $(n+1)$-th token. The model selects the index with the highest logit: $y = w_{n+1}$.

## 2. Data Processing
* **Dataset:** WikiText-2-raw-v1 (10,000 samples for training).
* **Cleaning Steps:**
    * **Normalization:** Convert all text to lowercase.
    * **Noise Removal:** Use regular expressions to remove special characters and punctuation, retaining only alphanumeric tokens.
    * **Tokenization:** Split cleaned strings into individual words.
    * **Vocab Construction:** Build a mapping for the top 10,000 frequent words, plus `<PAD>` and `<UNK>` tokens. Total **Vocab Size: 10,002**.

## 3. Model Architecture
The model is a decoder-only style **Mini Transformer** built with the following parameters:
* **Embedding Layer:** Maps token IDs to a $d_{model} = 256$ vector space.
* **Positional Encoding:** A learned embedding layer injecting sequence order for a `MAX_LEN` of 64 tokens.
* **Transformer Encoder Layers (x4):**
    * **Multi-Head Self-Attention:** Utilizes **8 heads** to attend to different representation subspaces.
    * **Causal Masking:** A triangular upper mask ensures tokens only attend to past/current positions.
    * **Feed-Forward Network:** Position-wise MLP with a hidden dimension of 512.
    * **Add & Norm:** Residual connections and Layer Normalization.
* **Language Modeling Head:** Projects output back to the 10,002 vocabulary size.



## 4. Results
* **Training Configuration:** AdamW ($LR=3e-4$, $\beta=(0.9, 0.98)$) and `ReduceLROnPlateau` scheduler.
* **Training Logs:**
| Epoch | Train Loss | Val Loss | Val Acc | Status |
| :--- | :--- | :--- | :--- | :--- |
| 1 | 6.5655 | 6.0413 | 15.28% | Saved Best Model |
| 5 | 5.8795 | **5.7947** | 17.62% | **Best Val Loss** |
| 10 | 7.1826 | 5.9654 | 18.82% | Early Stopping Triggered |

* **Final Test Performance:**
    * **Test Loss:** 5.7263
    * **Test Accuracy:** **18.00%**

## 5. Error Analysis
* **Successful Contextual Prediction:** In **Sentence 5**, the model correctly predicted "was" following "...music video for the song", showing it learned common English structures.
* **High-Frequency Bias:** In **Sentence 3**, the model predicted "the" instead of "having", likely due to the high probability density of "the" in the corpus.
* **Out-of-Vocabulary (OOV) Issues:** Predictions for **Sentences 2, 6, 7, and 10** resulted in `<UNK>`. This indicates the training subset (10k lines) was insufficient to learn specialized words like "whishaw" or "lewis".
* **Syntactic Confusion:** In **Sentence 1**, the model predicted "were" instead of "for", suggesting it grasped the need for a verb or preposition but picked a high-probability auxiliary verb instead.

## 6. Improvements
* **Increase Sequence Length:** While increased to 64, further expanding to 128 or 256 would allow the model to capture context across multiple sentences.
* **Deeper Architecture:** Increasing `NUM_LAYERS` to 6 or 8 would help in learning more complex linguistic patterns.
* **Larger Dataset:** Using the full WikiText dataset instead of 10,000 samples would significantly reduce the occurrence of `<UNK>` predictions.

## 7. Conclusion
* **Learning:** I successfully implemented the Transformer architecture, emphasizing the importance of causal masking and positional encoding for parallel sequence processing.
* **Effectiveness:** The **AdamW** optimizer and **ReduceLROnPlateau** scheduler effectively stabilized training, leading to a respectable 18% accuracy.
* **Takeaway:** Early stopping proved vital in this experiment, saving the best version of the model at Epoch 5 before the validation loss began to diverge from the training loss.